In [ ]:
from module import StrokeDataTransformer, df_test, df_prefit

import joblib
import pandas as pd
from sklearn.pipeline import Pipeline


voting  = joblib.load('models/best_voting_soft_0.2_0.0_0.8.joblib')

pipeline = Pipeline([
    ('preprocessor', StrokeDataTransformer()),
    ('classifier', voting)
])

test_ids = df_test['id']
X_test = df_test.drop('id', axis=1)

pipeline.named_steps['preprocessor'].fit(df_prefit)

proba = pipeline.predict_proba(X_test)[:, 1]
pred = pipeline.predict(X_test)

submission = pd.DataFrame({
    'id': test_ids,
    'stroke': pred
})

submission.to_csv('submission_test_data.csv', index=False)

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    roc_auc_score,
    average_precision_score
)


true_df = pd.read_csv('healthcare-sub.csv')
test_sub = pd.read_csv('submission_test_data.csv')

assert (test_sub['id'] == true_df['id']).all(), 'id не совпадают'

y_true = true_df['stroke'].values
y_proba = test_sub['stroke'].values
y_pred = proba

accuracy = accuracy_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_proba)
pr_auc = average_precision_score(y_true, y_proba)

metrics = pd.DataFrame({
    'Metric': ['Accuracy', 'Recall', 'ROC-AUC', 'PR-AUC'],
    'Value': [accuracy, recall, roc_auc, pr_auc]
})

In [13]:
print(metrics)

     Metric     Value
0  Accuracy  0.951487
1    Recall  0.000000
2   ROC-AUC  0.732584
3    PR-AUC  0.096105
